# **Taller 2: Cadenas de Markov**

### **Introducción: Procesamiento de Lenguaje Natural**
Un campo muy explorado de la inteligencia artificial es el procesamiento de lenguaje natural o NLP (Natural Language Processing), esta área estudia el lenguaje humano y sus posibles interacciones con la computación. Se utiliza la inteligencia artificial para representar y analizar el lenguaje con el fin de crear aplicaciones que puedan recrear este lenguaje par diversos fines.

En este taller, utilizaremos las cadenas de Markov para intentar recrear la estructura del texto, intentando predecir que palabras pueden suceder a otras y de esta forma producir textos con lenguaje natural, una técnica muy utilizada en los teclados de los smartphones y en los editores de texto.

### **Los datos:**
En este caso, contamos con un conjunto de datos de texto plano, dado que el texto no se puede recrear como input o output, solo contamos con los archivos de texto los cuales serán leídos para recrearlos. El texto con el que contamos corresponde a distintas historias de Sherlock Holmes, el famoso detective de ficción creado por Arthur Conan Doyle.

---

Cargamos algunas de las librerías necesarias para este taller

In [185]:
import numpy as np
import pandas as pd
import math
import os
import re
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
import random
import codecs


A continuación se presenta el código mediante el cual se hace la lectura de los archivos de texto y su respectivo preprocesamiento y limpieza, no se debe modificar.

In [186]:
def read_all_stories(story_path):
    '''
    Retorna una lista con todas las líneas de cada uno de los archivos de texto en el orden en el que están escritas.
    
    Parametros:
        story_path: la direccion de la carpeta en la que estan los archivos a leer
    Retorna:
        txt: una lista con las lineas de los archivos
    '''
    txt = []
    for _, _, files in os.walk(story_path):
        for file in files:
            with codecs.open(story_path+file) as f:
                for line in f:
                    line = line.strip()
                    if line=='----------': break
                    if line!='':txt.append(line)
    return txt

In [187]:
def clean_txt(txt):
    '''
    Retorna una lista con el texto preprocesado.
    
    Parametros:
        txt: una lista con lineas de texto
    Retorna:
        cleaned_txt: una lista con todas las palabras luego de retirar caracteres especiales y separarlas
    '''
    cleaned_txt = []
    for line in txt:
        line = line.lower()
        line = re.sub(r"[,.\"\'!@#$%^&*(){}?/;`~:<>+=-\\]", "", line)
        tokens = word_tokenize(line)
        words = [word for word in tokens if word.isalpha()]
        cleaned_txt+=words
    return cleaned_txt

In [188]:
story_path = "data/sherlock/"
stories = read_all_stories(story_path)
cleaned_stories = clean_txt(stories)

### **Creación del modelo de Markov:**
A continuación se presenta la función ```make_markov_model``` la cual crea nuestro modelo de Markov.
    
Esta función retorna la matriz de probabilidades del modelo como un diccionario de Python. Las probabilidades han sido calculadas teniendo en cuenta la cantidad de veces que aparece una palabra o una secuencia de palabras [n-grama](https://en.wikipedia.org/wiki/N-gram) seguida de otra.

In [189]:
def make_markov_model(cleaned_txt, n_gram=2):
    markov_model = {}
    for i in range(len(cleaned_txt)-n_gram-1):
        curr_state, next_state = "", ""
        for j in range(n_gram):
            curr_state += cleaned_txt[i+j] + " "
            next_state += cleaned_txt[i+j+n_gram] + " "
        curr_state = curr_state[:-1]
        next_state = next_state[:-1]
        if curr_state not in markov_model:
            markov_model[curr_state] = {}
            markov_model[curr_state][next_state] = 1
        else:
            if next_state in markov_model[curr_state]:
                markov_model[curr_state][next_state] += 1
            else:
                markov_model[curr_state][next_state] = 1
    
    for curr_state, transition in markov_model.items():
        total = sum(transition.values())
        for state, count in transition.items():
            markov_model[curr_state][state] = count/total
        
    return markov_model

1. Crear el modelo de Markov utilizando los datos extraídos y mostrar la cantidad de estados de la matriz de probabilidad, mantener el valor de los n-gramas en 2 (bigramas: duos de palabras)

In [190]:
markov_model = make_markov_model(cleaned_stories, n_gram=2)
cantidad_estados = len(markov_model)
print(f"El modelo de Markov tiene {cantidad_estados} estados.")


El modelo de Markov tiene 208717 estados.


2. Generando texto. <br>
    A continuación se presenta la función ```generate_story``` la cual produce texto con base en un modelo de Markov, los parámetros se presentan a continuación: <br>
    - ```markov_model``` es el modelo de Markov. <br>
    - ```limit``` es el límite de palabras que se van a predecir. <br>
    - ```start``` es el valor de inicio de la cadena
    
    Para predecir los estados de la cadena se utilizará la función ```choices``` de la librería ```random``` de Python. Esta función recibe una lista sober la cual escoger un dato aleatorio y otra lista con los pesos de cada dato para realizar la elección, para más información ver la [documentación de Python](https://docs.python.org/3/library/random.html).
    
    Se debe completar esta función para que sea capaz de generar texto basándose enteramente en las probabilidades presentadas por el modelo de Markov, es decir, teniendo en cuenta el estado actual (palabra/palabras), se debe predecir el siguiente, y así sucesivamente hasta completar el límite.

In [191]:
def generate_story(markov_model, start, limit=100):
    curr_state = start.strip().lower()   # estado inicial normalizado
    story = [curr_state]                 # acumulamos en una lista para mejor manejo
    for _ in range(limit//len(start.split(" "))):
        # Si el estado actual no está en el modelo, se detiene
        if curr_state not in markov_model:
            break;

        # Posibles estados siguientes y sus probabilidades
        next_states = list(markov_model[curr_state].keys())
        probabilities = list(markov_model[curr_state].values())

        # Elegimos el siguiente estado según las probabilidades
        next_state = random.choices(next_states, weights=probabilities)[0]

        # Agregamos la predicción al texto
        story.append(next_state)

        # preparamos el siguiente estado actual
        curr_state = next_state

    return " ".join(story)


3. Crear 10 secuencias utilizando el modelo, un punto de inicio cualquiera y un limite de 10, ¿Tiene sentido el texto generado? <br>
Repetir este proceso 5 veces con distintos puntos de inicio.

In [192]:
start_phrases = [
   "the detective",
    "sherlock holmes",
    "i was",
    "my friend",
    "in the"
]

for start in start_phrases:
    print(f"\n===== Punto de inicio: '{start}' =====\n")
    for i in range(10):
        story = generate_story(markov_model, start, limit=10)
        print(f"Secuencia {i+1}: {story}")
    print("\n")



===== Punto de inicio: 'the detective' =====

Secuencia 1: the detective was to lead to further developments which may assist you
Secuencia 2: the detective indifferent and contemptuous while i watched his son was exaggerating
Secuencia 3: the detective now that you mention it thick it is not difficult
Secuencia 4: the detective really mr holmes i fear there is not quite the
Secuencia 5: the detective yes the actual facts are very quick at observing that
Secuencia 6: the detective shook hands eagerly with sherlock holmes why it is the
Secuencia 7: the detective he sank back into his chair and upset the remainder
Secuencia 8: the detective answered proudly the young mans last words the character of
Secuencia 9: the detective really mr holmes this is my friend and i the
Secuencia 10: the detective line and their weakness both in his gifts and in



===== Punto de inicio: 'sherlock holmes' =====

Secuencia 1: sherlock holmes laughed heartily at my friend like a man and of
Secuencia 2: sh

4. Construir un nuevo modelo de Markov utilizando únicamente palabras (n_grams=1), imprimir la cantidad de estados con los que cuenta esta nueva matriz.

In [193]:
markov_model2 = make_markov_model(cleaned_stories, n_gram=1)
cantidad_estados = len(markov_model2)
print(f"El modelo de Markov tiene {cantidad_estados} estados.")

El modelo de Markov tiene 19017 estados.


5. Repetir el punto 4, esta vez utilizando el nuevo modelo, dejar el límite en 15. ¿Hay variación en el texto generado en este caso con respecto al anterior?

In [194]:

start_phrases = [
    "detective",
    "sherlock",
    "i",
    "my",
    "in"
]

for start in start_phrases:
    print(f"\n===== Punto de inicio: '{start}' =====\n")
    for i in range(10):
        story = generate_story(markov_model2, start, limit=15)
        print(f"Secuencia {i+1}: {story}")
    print("\n")


===== Punto de inicio: 'detective' =====

Secuencia 1: detective i noted and would he is by the central position but holmes held his appearance
Secuencia 2: detective he says have not hurt said he loved the found him the skin for but
Secuencia 3: detective seated in the ground that i must slip of his eyes glowed and it into
Secuencia 4: detective service he was an hour of a story and surprise to his house i think
Secuencia 5: detective really difficult to admit a most unjustifiable intrusion of my astonished detective looked up and
Secuencia 6: detective if there yes the man were however and as i could i nearly five foot
Secuencia 7: detective was the wall at last fortnight for the bicycle in its appearance i think that
Secuencia 8: detective he understood its way to my thoughts in the way when i said if i
Secuencia 9: detective and bided my own misfortune came out into difficulties let you could succeed where his
Secuencia 10: detective i i cried why had recovered you budge from his 

6. Utilizando el modelo que haya dado un mejor resultado según su opinión, crear un texto de al menos 100 palabras, ¿Tiene sentido este texto? ¿Que hace falta para dar más coherencia al texto?

In [195]:
story = generate_story(markov_model, "my notebook", limit=100)
print(f"Usando modelo 1: {story}")

story = generate_story(markov_model2, "detective", limit=100)
print(f"Usando modelo 2: {story}")

Usando modelo 1: my notebook that it first informed me that was where i had a fit of anger and sprang upon the moor when he was quite dead said he the duke and mr alec was smoking a cigar he softly clapped his fat hands and looking from one to colonel ross as well as in devonshire a devil with the slack season i am a little upset and you will devote his whole theory is that he stood beside me in them and he tried to bluster and took a nice little grove of scotch firs and i used money which i had
Usando modelo 2: detective someone in a fellow has been driven home and looking for i cant you have you mean it has passed through the real name the rate i was all alarm those who had stood for white to the true enough of boss oh if you seen half up her elbow hullo chummy said holmes the piece of a curious facts and gazed at her husband her some six years living voice died was as if these candles and dangerous guest and mocking smile broadened as he tell mrs douglas may make the room at your th

*El modelo 1*

El modelo uno resulta más coherente porque conserva elementos narrativos propios de Conan Doyle: diálogos atribuidos a Holmes, giros lingüísticos victorianos (“I should be very much obliged”), descripciones de gestos y escenarios, y referencias a lugares auténticos del canon (Bristol, Bay of Bengal, Port Blair). Aunque desordenado, transmite acción y contexto reconocibles, lo que evoca mejor la atmósfera detectivesca de Sherlock. El modelo dos, en cambio, mezcla nombres reales del canon con frases inconexas y sintaxis rota, perdiendo continuidad narrativa.

7. Utilizando las funciones dadas para leer y limpiar texto, crear un nuevo modelo de Markov, esta vez utilizando texto en español, se recomienda utilizar una cantidad elevada de texto, algunos ejemplos de datos que se pueden tomar pueden ser libros reconocidos, o distintos articulos web como los articulos de wikipedia. Repetir los puntos 4 al 7 utilizando el mismo código pero los datos en español, ¿El texto es más coherente?.

In [198]:
story_file= "./data/cien_anos_soledad.txt"
txt = []
with codecs.open(story_file, encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if line == '----------':  
            break
        if line != '':
            txt.append(line)

cleaned_stories = clean_txt(txt)

markov_model3 = make_markov_model(cleaned_stories, n_gram=2)
cantidad_estados = len(markov_model3)
print(f"El modelo de Markov tiene {cantidad_estados} estados.")


start_phrases = [
    "muchos años",        
    "coronel aureliano",  
    "josé arcadio",       
    "macondo era",        
    "remedios la"
]


for start in start_phrases:
    print(f"\n===== Punto de inicio: '{start}' =====\n")
    for i in range(10):
        story = generate_story(markov_model3, start, limit=30)
        print(f"Secuencia {i+1}: {story}")
    print("\n")            

El modelo de Markov tiene 69028 estados.

===== Punto de inicio: 'muchos años' =====

Secuencia 1: muchos años más tarde para conocer el lugar más profundo del mar en una madrugada de la ejecución debía cumplirse en el cuarto de melquíades estaba en el cuarto santa sofía de
Secuencia 2: muchos años que habían vendido la guerra por un aposento invadido por la maleza y arropadas con piltrafas de guirnaldas donde las macilentas y gordas viudas de nadie donde en épocas pasadas
Secuencia 3: muchos años hubo aquí una calle adyacente la posición privilegiada con respecto a la aldea durante varias semanas encerrado en un cuartito con puerta hacia la puerta trató de sonreír tomó los
Secuencia 4: muchos años más tarde le hizo caso sino que se lo dejaran en paz descanse tuviera el atrevimiento do preguntar con su antigua voz estentórea restaurada por un tropel de adolescentes incansables
Secuencia 5: muchos años el aire de juventud y de miedo y el ansia atolondrada de huir y al mismo tiempo tan d